# MIR — DinoV2 (Zero-Shot + Finetuning SupCon)

Pipeline :
1. Zero-shot DinoV2 (extraction + Recall@K + mAP)
2. Fine-tuning avec SupConLoss + MultiSimilarityMiner sur une tête de projection (backbone gelé)
3. Suivi train/val loss + Val Recall@1 par epoch
4. Évaluation après finetuning + comparaison Zero-Shot vs SupCon

Tout est sauvegardé dans `MIR_Project/DinoV2_training` sur Drive.

In [ ]:
!pip install timm==1.0.15 pytorch-metric-learning --quiet

In [ ]:
import os
import json
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from pytorch_metric_learning import losses, miners, samplers

# Chemins
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGE_DIR    = os.path.join(PROJECT_ROOT, "dataset")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "results", "DinoV2_training")
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device       : {device}')
print(f'Dataset      : {IMAGE_DIR}')
print(f'Sauvegardes  : {SAVE_DIR}')
print(f'GPU          : {os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip()}')

In [ ]:
def get_label_from_filename(filename):
    """Format attendu : 0_3_BMW_Serie5_474.jpg -> 'BMW_Serie5'."""
    name_without_ext = filename.split('.')[0]
    parts = name_without_ext.split('_')
    if len(parts) >= 4:
        return f'{parts[2]}_{parts[3]}'
    return 'Label_Inconnu'


class SimpleImageDataset(Dataset):
    """Retourne (image, filename) — utilisé pour l'extraction d'embeddings."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_filenames = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
        ]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_filenames[idx]


# Transformation standard d'inférence pour DinoV2
eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print('Helpers OK.')

In [ ]:
def evaluate_retrieval(embeddings, labels, K_values=(1, 5, 20, 50)):
    """Calcule Recall@K et mAP. embeddings : (N, D) L2-normalisés."""
    K_values = list(K_values)
    recalls     = {k: 0 for k in K_values}
    ap_scores   = []
    num_queries = len(embeddings)

    similarity_matrix = torch.mm(embeddings, embeddings.T)

    for i in tqdm(range(num_queries), desc='Eval'):
        query_label = labels[i]
        scores      = similarity_matrix[i].clone()
        scores[i]   = -1.0

        _, top_indices   = torch.topk(scores, max(K_values))
        retrieved_labels = [labels[idx.item()] for idx in top_indices]

        for k in K_values:
            if query_label in retrieved_labels[:k]:
                recalls[k] += 1

        num_relevant = labels.count(query_label) - 1
        if num_relevant == 0:
            continue

        hits, precision_sum = 0, 0.0
        for rank, lbl in enumerate(retrieved_labels, start=1):
            if lbl == query_label:
                hits += 1
                precision_sum += hits / rank
        ap_scores.append(precision_sum / num_relevant)

    recall_scores = {k: recalls[k] / num_queries * 100 for k in K_values}
    mAP           = sum(ap_scores) / len(ap_scores) * 100
    return recall_scores, mAP


def plot_and_save_recall(recall_scores, mAP, title, save_basename, color='#1f77b4', marker='o'):
    """Trace la courbe Recall@K et sauvegarde graphe + JSON."""
    K_values = list(recall_scores.keys())

    plt.figure(figsize=(10, 6))
    plt.plot(K_values, list(recall_scores.values()),
             marker=marker, linestyle='-', color=color,
             linewidth=2.5, markersize=8, label='DinoV2')

    for k, v in recall_scores.items():
        plt.annotate(f'{v:.1f}%', (k, v),
                     textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=10)

    plt.text(0.98, 0.05, f'mAP : {mAP:.2f}%',
             transform=plt.gca().transAxes, fontsize=11,
             ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
    plt.ylabel('Taux de Rappel (%)', fontsize=12)
    plt.xticks(K_values)
    plt.ylim(0, 105)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()

    graph_path  = os.path.join(SAVE_DIR, f'{save_basename}.png')
    result_path = os.path.join(SAVE_DIR, f'results_{save_basename}.json')

    plt.savefig(graph_path, dpi=300, bbox_inches='tight')
    results = {
        'model':    save_basename,
        'recall':   {f'@{k}': round(v, 2) for k, v in recall_scores.items()},
        'mAP':      round(mAP, 2),
    }
    with open(result_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f'Graphique : {graph_path}')
    print(f'Résultats : {result_path}')
    plt.show()
    return results

print('Eval helpers OK.')

## 1. DinoV2 Zero-Shot

In [ ]:
model_zs = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(device)
model_zs.eval()
print(f'DinoV2 ViT-B/14 chargé. Dimension de sortie : 768')

In [ ]:
dataset    = SimpleImageDataset(root_dir=IMAGE_DIR, transform=eval_transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4)

all_embeddings = []
all_filenames  = []

print(f'Extraction des vecteurs pour {len(dataset)} images...')
with torch.no_grad():
    for images, filenames in tqdm(dataloader):
        images = images.to(device)
        embeddings = model_zs(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
        all_filenames.extend(filenames)

database_tensor = torch.cat(all_embeddings, dim=0)

save_path = os.path.join(SAVE_DIR, 'DinoV2_zero_shot.pth')
torch.save({'embeddings': database_tensor, 'filenames': all_filenames}, save_path)
print(f'Base de données : {database_tensor.shape} → {save_path}')

In [ ]:
db         = torch.load(os.path.join(SAVE_DIR, 'DinoV2_zero_shot.pth'))
embeddings = db['embeddings']
filenames  = db['filenames']
labels     = [get_label_from_filename(f) for f in filenames]

recall_scores, mAP = evaluate_retrieval(embeddings, labels)

print('\n--- Résultats Zero-Shot DinoV2 ---')
for k, v in recall_scores.items():
    print(f'Recall@{k:>2} : {v:.2f}%')
print(f'mAP        : {mAP:.2f}%')

plot_and_save_recall(recall_scores, mAP,
                     title='Courbe de Rappel (Recall@K) - DinoV2 zero shot',
                     save_basename='recall_curve_DinoV2_zero_shot',
                     color='#1f77b4', marker='o')

## 2. DinoV2 Fine-tuning — SupConLoss

Backbone DinoV2 gelé, tête de projection 768×2 → 256 entraînée avec **SupConLoss** + **MultiSimilarityMiner**. Split stratifié train/val 85/15 pour suivre la dynamique d'entraînement (train loss, val loss, val Recall@1).

In [ ]:
class LabeledImageDataset(Dataset):
    """Retourne (image, label_int) — pour les losses metric learning."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_filenames = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
        ]
        all_labels    = [get_label_from_filename(f) for f in self.image_filenames]
        unique_labels = sorted(set(all_labels))
        self.label2idx = {l: i for i, l in enumerate(unique_labels)}
        self.targets   = [self.label2idx[l] for l in all_labels]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.targets[idx]


# Augmentations pour le train
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Datasets : mêmes images, transforms différents
full_train_ds = LabeledImageDataset(root_dir=IMAGE_DIR, transform=train_transform)
full_eval_ds  = LabeledImageDataset(root_dir=IMAGE_DIR, transform=eval_transform)
NUM_CLASSES   = len(full_train_ds.label2idx)

# Split stratifié 85/15 par classe
VAL_RATIO = 0.15
random.seed(42)

by_class = defaultdict(list)
for idx, t in enumerate(full_train_ds.targets):
    by_class[t].append(idx)

train_indices, val_indices = [], []
for cls, idxs in by_class.items():
    random.shuffle(idxs)
    n_val = max(1, int(len(idxs) * VAL_RATIO))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

train_subset = Subset(full_train_ds, train_indices)
val_subset   = Subset(full_eval_ds,  val_indices)

train_targets = [full_train_ds.targets[i] for i in train_indices]

print(f'Classes : {NUM_CLASSES}')
print(f'Train   : {len(train_subset)} images')
print(f'Val     : {len(val_subset)} images')

In [ ]:
class CarRetrievalModel(nn.Module):
    """
    DinoV2 ViT-B/14 gelé → concat(CLS, mean(patch_tokens)) → tête MLP → embedding 256-D.
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.proj = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.1),
            nn.Linear(512, embed_dim),
        )

    def extract_features(self, x):
        with torch.no_grad():
            feats = self.backbone.forward_features(x)
            cls   = feats['x_norm_clstoken']
            patch = feats['x_norm_patchtokens'].mean(dim=1)
        return torch.cat([cls, patch], dim=-1)  # (B, 1536)

    def forward(self, x):
        emb = self.extract_features(x)
        return self.proj(emb)


EMBED_DIM = 256
model = CarRetrievalModel(embed_dim=EMBED_DIM).to(device)

n_trainable = sum(p.numel() for p in model.proj.parameters())
print(f'CarRetrievalModel instancié.')
print(f'Paramètres entraînables (tête proj) : {n_trainable:,}')

In [ ]:
# Sampler stratifié par classe : garantit des positifs dans chaque batch (clé pour SupCon)
sampler = samplers.MPerClassSampler(
    labels=train_targets,
    m=8,
    batch_size=128,
    length_before_new_iter=len(train_subset),
)

train_loader = DataLoader(
    train_subset, batch_size=128,
    sampler=sampler, num_workers=4, drop_last=True,
)

val_loader = DataLoader(
    val_subset, batch_size=128,
    shuffle=True, num_workers=4, drop_last=False,
)

loss_fn   = losses.SupConLoss(temperature=0.07)
miner     = miners.MultiSimilarityMiner(epsilon=0.1)

optimizer = optim.AdamW(
    model.proj.parameters(),
    lr=1e-3,
    weight_decay=0.05,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

print(f'Batches train : {len(train_loader)} | Batches val : {len(val_loader)}')
print(f'Loss : SupConLoss(temperature=0.07) + MultiSimilarityMiner(epsilon=0.1)')

In [ ]:
best_recall = 0
patience    = 10
no_improve  = 0
NUM_EPOCHS  = 30

best_model_path = os.path.join(SAVE_DIR, 'DinoV2_SupCon_finetuned.pth')

history = {
    'epoch':        [],
    'train_loss':   [],
    'val_loss':     [],
    'val_recall@1': [],
    'lr':           [],
}

for epoch in range(NUM_EPOCHS):
    # ============ TRAIN ============
    model.train()
    train_loss_sum = 0.0
    n_train_batches = 0

    for images, lbls in tqdm(train_loader, desc=f'Epoch {epoch+1} [train]'):
        images = images.to(device)
        lbls   = lbls.to(device)

        embeddings = model(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)

        hard_pairs = miner(embeddings, lbls)
        loss       = loss_fn(embeddings, lbls, hard_pairs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        n_train_batches += 1

    scheduler.step()
    avg_train_loss = train_loss_sum / max(n_train_batches, 1)

    # ============ VALIDATION ============
    model.eval()
    val_loss_sum = 0.0
    n_val_batches = 0
    all_embs, all_lbls = [], []

    with torch.no_grad():
        for images, lbls in tqdm(val_loader, desc=f'Epoch {epoch+1} [val]  '):
            images = images.to(device)
            lbls   = lbls.to(device)

            embeddings = model(images)
            embeddings = F.normalize(embeddings, p=2, dim=1)

            v_loss = loss_fn(embeddings, lbls)  # sans miner en val
            val_loss_sum += v_loss.item()
            n_val_batches += 1

            all_embs.append(embeddings.cpu())
            all_lbls.extend(lbls.cpu().tolist())

    avg_val_loss = val_loss_sum / max(n_val_batches, 1)

    # Recall@1 sur le val set
    all_embs = torch.cat(all_embs)
    sim      = torch.mm(all_embs, all_embs.T)
    correct  = 0
    for i in range(len(all_embs)):
        scores     = sim[i].clone()
        scores[i]  = -1.0
        top1_label = all_lbls[torch.argmax(scores).item()]
        if top1_label == all_lbls[i]:
            correct += 1

    val_recall_at_1 = correct / len(all_embs) * 100
    current_lr      = optimizer.param_groups[0]['lr']

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_recall@1'].append(val_recall_at_1)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch+1:>3} | '
          f'Train Loss {avg_train_loss:.4f} | '
          f'Val Loss {avg_val_loss:.4f} | '
          f'Val Recall@1 {val_recall_at_1:.1f}% | '
          f'LR {current_lr:.2e}')

    # ============ CHECKPOINT + EARLY STOPPING ============
    if val_recall_at_1 > best_recall:
        best_recall = val_recall_at_1
        torch.save({
            'model_state':   model.state_dict(),
            'label2idx':     full_train_ds.label2idx,
            'embed_dim':     EMBED_DIM,
            'best_recall@1': val_recall_at_1,
            'epoch':         epoch + 1,
        }, best_model_path)
        no_improve = 0
        print(f'  ✓ Nouveau meilleur modèle (Val Recall@1 = {val_recall_at_1:.1f}%)')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping à l'epoch {epoch+1}")
            break

history_path = os.path.join(SAVE_DIR, 'training_history_SupCon.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nMeilleur Val Recall@1 : {best_recall:.1f}%')
print(f'Modèle sauvegardé   : {best_model_path}')
print(f'Historique sauvé    : {history_path}')

In [ ]:
with open(os.path.join(SAVE_DIR, 'training_history_SupCon.json')) as f:
    history = json.load(f)

epochs     = history['epoch']
train_loss = history['train_loss']
val_loss   = history['val_loss']
val_recall = history['val_recall@1']

best_idx    = max(range(len(val_recall)), key=lambda i: val_recall[i])
best_epoch  = epochs[best_idx]
best_recall = val_recall[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sous-graphe 1 : Loss train vs val
ax = axes[0]
ax.plot(epochs, train_loss, marker='o', linewidth=2, markersize=6,
        color='#1f77b4', label='Train Loss')
ax.plot(epochs, val_loss,   marker='s', linewidth=2, markersize=6,
        color='#d62728', label='Validation Loss')
ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7,
           label=f'Best epoch ({best_epoch})')
ax.set_title('Courbes de Loss — DinoV2 SupCon', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xticks(epochs)

# Sous-graphe 2 : Recall@1 validation
ax = axes[1]
ax.plot(epochs, val_recall, marker='D', linewidth=2, markersize=6,
        color='#2ca02c', label='Val Recall@1')
ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7)
ax.axhline(best_recall, color='gray', linestyle=':', alpha=0.5)
ax.annotate(f'Best : {best_recall:.1f}% @ epoch {best_epoch}',
            xy=(best_epoch, best_recall),
            xytext=(10, -20), textcoords='offset points',
            fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Recall@1 sur le set de validation', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Recall@1 (%)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xticks(epochs)
ax.set_ylim(0, 105)

plt.tight_layout()
curves_path = os.path.join(SAVE_DIR, 'training_curves_DinoV2_SupCon.png')
plt.savefig(curves_path, dpi=300, bbox_inches='tight')
print(f'Courbes sauvegardées : {curves_path}')
plt.show()

print('\n--- Résumé de l\'entraînement ---')
print(f'Epochs réalisés   : {len(epochs)}')
print(f'Train loss final  : {train_loss[-1]:.4f}')
print(f'Val loss final    : {val_loss[-1]:.4f}')
print(f'Meilleur Recall@1 : {best_recall:.1f}% (epoch {best_epoch})')

## 3. Évaluation du modèle finetuné sur toute la base

In [ ]:
checkpoint = torch.load(os.path.join(SAVE_DIR, 'DinoV2_SupCon_finetuned.pth'))
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Modèle finetuné chargé (best Val Recall@1 = {checkpoint['best_recall@1']:.1f}% à l'epoch {checkpoint['epoch']}).")

In [ ]:
dataset_eval    = SimpleImageDataset(root_dir=IMAGE_DIR, transform=eval_transform)
dataloader_eval = DataLoader(dataset_eval, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4)

all_embeddings, all_filenames = [], []

print(f'Extraction des vecteurs pour {len(dataset_eval)} images...')
with torch.no_grad():
    for images, filenames in tqdm(dataloader_eval):
        images = images.to(device)
        embeddings = model(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
        all_filenames.extend(filenames)

database_tensor = torch.cat(all_embeddings, dim=0)

save_path_ft = os.path.join(SAVE_DIR, 'DinoV2_SupCon_finetuned_best.pth')
torch.save({'embeddings': database_tensor, 'filenames': all_filenames}, save_path_ft)
print(f'Base de données : {database_tensor.shape} → {save_path_ft}')

In [ ]:
db         = torch.load(os.path.join(SAVE_DIR, 'DinoV2_SupCon_finetuned_best.pth'))
embeddings = db['embeddings']
filenames  = db['filenames']
labels     = [get_label_from_filename(f) for f in filenames]

recall_scores, mAP = evaluate_retrieval(embeddings, labels)

print('\n--- Résultats DinoV2 SupCon Finetuned ---')
for k, v in recall_scores.items():
    print(f'Recall@{k:>2} : {v:.2f}%')
print(f'mAP        : {mAP:.2f}%')

plot_and_save_recall(recall_scores, mAP,
                     title='Courbe de Rappel (Recall@K) - DinoV2 SupCon Finetuned',
                     save_basename='recall_curve_DinoV2_SupCon',
                     color='#2ca02c', marker='s')

## 4. Comparaison Zero-Shot vs SupCon Finetuned

In [ ]:
with open(os.path.join(SAVE_DIR, 'results_recall_curve_DinoV2_zero_shot.json')) as f:
    zero_shot = json.load(f)
with open(os.path.join(SAVE_DIR, 'results_recall_curve_DinoV2_SupCon.json')) as f:
    supcon = json.load(f)

K_values      = [1, 5, 20, 50]
scores_zero   = [zero_shot['recall'][f'@{k}'] for k in K_values]
scores_supcon = [supcon['recall'][f'@{k}']    for k in K_values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_values, scores_zero,   marker='o', linewidth=2.5, markersize=8,
        color='#1f77b4',
        label=f"Zero-Shot (mAP {zero_shot['mAP']}%)")
ax.plot(K_values, scores_supcon, marker='s', linewidth=2.5, markersize=8,
        color='#2ca02c',
        label=f"SupCon Finetuned (mAP {supcon['mAP']}%)")

for k, v in zip(K_values, scores_zero):
    ax.annotate(f'{v:.1f}%', (k, v), textcoords='offset points',
                xytext=(-15, 6), fontsize=9)
for k, v in zip(K_values, scores_supcon):
    ax.annotate(f'{v:.1f}%', (k, v), textcoords='offset points',
                xytext=(5, 6), fontsize=9)

ax.set_title('Recall@K — DinoV2 Zero-Shot vs SupCon Finetuned',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
ax.set_ylabel('Taux de Rappel (%)', fontsize=12)
ax.set_xticks(K_values)
ax.set_ylim(0, 105)
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(fontsize=11)

plt.tight_layout()
save_path = os.path.join(SAVE_DIR, 'comparaison_recall_DinoV2_SupCon.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'Comparaison sauvegardée : {save_path}')
plt.show()